# PDESolve overview

Every example calls `pdesolve(...)` and demonstrates the package entry point across several solver families.

## Methods highlighted here
- `auto`
- `constant_coefficient_inverse_operator`
- `symmetry_reduction`
- `charpit`
- `complete_integral`
- `invariant_reduction_auto`
- `unified_transform`
- `hyperbolic_system`

In [ ]:
import sympy as sp
import pdesolve as pds
x, y, z, t = sp.symbols('x y z t', real=True)
u = sp.Function('u')

## Structured condition and geometry planning

`pdesolve(...)` now normalizes bundled IC/BC input into a `ConditionModel` and infers a `DomainGeometry` object that the planner uses directly.

In [ ]:
problem = pds.build_pde_problem(
    sp.Eq(sp.diff(u(x, t), t), sp.diff(u(x, t), x, 2)),
    u(x, t),
    (x, t),
    ics=[sp.Eq(u(x, 0), sp.sin(x))],
    bcs=[sp.Eq(u(0, t), 0), sp.Eq(u(sp.pi, t), 0)],
)
problem.canonical_representation.details['condition_summary'], problem.canonical_representation.details['domain_summary']

## Auto planner

In [ ]:
eq_auto = sp.Eq(sp.diff(u(x, t), t) + sp.diff(u(x, t), x), 0)
pds.pdesolve(eq_auto, u(x, t), (x, t), method='auto')

## Constant-coefficient inverse differential operator

In [ ]:
ic_eq = sp.Eq(u(x, 0), sp.exp(-x**2))
pds.pdesolve(eq_auto, u(x, t), (x, t), method='constant_coefficient_inverse_operator', ics={'equation': ic_eq, 'initial_profile': sp.exp(-x**2), 'curve_value': 0})

## Symmetry reduction

In [ ]:
pds.pdesolve(eq_auto, u(x, t), (x, t), method='symmetry_reduction')

## Charpit

In [ ]:
eq_char = sp.Eq(sp.diff(u(x, y), x)**2 + sp.diff(u(x, y), y)**2, 1)
pds.pdesolve(eq_char, u(x, y), (x, y), method='charpit')

## Complete integral

In [ ]:
eq_ci = sp.Eq(sp.diff(u(x, y), x)**2 + sp.diff(u(x, y), y), 0)
pds.pdesolve(eq_ci, u(x, y), (x, y), method='complete_integral')

## Invariant reduction

In [ ]:
eq_inv = sp.Eq(sp.diff(u(x, t), t) + u(x, t) * sp.diff(u(x, t), x), 0)
pds.pdesolve(eq_inv, u(x, t), (x, t), method='invariant_reduction_auto')

## Unified transform

In [ ]:
eq_s = sp.Eq(sp.I * sp.diff(u(x, t), t) + sp.diff(u(x, t), x, 2), 0)
bc_eqs = [sp.Eq(u(0, t), 0)]
pds.pdesolve(eq_s, u(x, t), (x, t), method='unified_transform', ics=ic_eq, bcs={'equations': bc_eqs}, domain='half_line')

## Hyperbolic system

In [ ]:
u1 = sp.Function('u1')
u2 = sp.Function('u2')
eqs = [
    sp.Eq(sp.diff(u1(t, x), t), sp.diff(u2(t, x), x)),
    sp.Eq(sp.diff(u2(t, x), t), sp.diff(u1(t, x), x)),
]
ics_sys = [sp.Eq(u1(0, x), sp.sin(x)), sp.Eq(u2(0, x), sp.cos(x))]
pds.pdesolve(eqs, (u1, u2), (t, x), method='hyperbolic_system', ics=ics_sys)

## Structured planning, geometry, and transforms

Recent versions of `pdesolve` expose structured planning objects directly. The canonical representation now carries a `ConditionModel`, `DomainGeometry`, `BoundaryModel`, `SeparableGeometryPlan`, and `TransformMethodPlan`. These are used by the planner and by the separation/series and transform solvers themselves.

In [ ]:
import sympy as sp
from pdesolve import build_pde_problem

x, t = sp.symbols("x t", real=True)
u = sp.Function("u")
problem = build_pde_problem(
    sp.Eq(sp.diff(u(x,t), t), sp.diff(u(x,t), x, 2)),
    u(x,t),
    (x, t),
    ics={"equation": sp.Eq(u(x,0), sp.sin(x))},
    bcs=[sp.Eq(u(0,t), 0), sp.Eq(u(sp.pi,t), 0)],
)
problem.details["condition_model"]
problem.details["domain_geometry"]
problem.details["boundary_model"]
problem.details["separation_plan"]
problem.details["transform_plan"]

## Structured execution paths

The demo now reflects that structured separation, transform, and first-order nonlinear methods execute directly from `ConditionModel`, `DomainGeometry`, and the canonical first-order PDE representation. In practical terms, interval/rectangle series solvers, transform methods, and generalized Clairaut detection can operate without falling back to raw-equation condition parsing in the common supported cases.

## Fundamental solutions and Green functions

The structured kernel subsystem can build classical heat, wave, and Laplace kernels directly from the canonical problem model.

In [ ]:
from sympy import symbols, Function, Eq, diff
from pdesolve import solve_fundamental_solution, solve_green_function

x, t, a, L = symbols('x t a L', positive=True)
u = Function('u')

heat_fs = solve_fundamental_solution(Eq(diff(u(x, t), t) - a*diff(u(x, t), x, 2), 0), u(x, t), (x, t))
heat_interval_green = solve_green_function(
    Eq(diff(u(x, t), t) - a*diff(u(x, t), x, 2), 0),
    u(x, t),
    (x, t),
    bcs=(Eq(u(0, t), 0), Eq(u(L, t), 0)),
)

heat_fs, heat_interval_green